In [1]:
# Used to test the existing model with "common sense problems" 
import sys
import os

sys.path.append(os.path.abspath(".."))
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import torch.optim as optim
import chess

from tqdm import tqdm
from torch.utils.data import DataLoader

from model import NNUE

from dataset import extract_halfkp
from engine.eval import model_evaluate_board

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(device)

mps


In [3]:
model = NNUE().to(device)
checkpoint = torch.load("./chess_model_final_120000.pt", map_location=torch.device('cpu')) # cuda trained needs this key
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [8]:
# test 1, give three fen positions, see if engine can evaluate reasonably
fen1 = "r1bqk1nr/pppp1ppp/2n5/2b1p1N1/2B1P3/8/PPPP1PPP/RNBQK2R b KQkq - 0 1" # normal opening position, should be near 0
print(model_evaluate_board(model, chess.Board(fen1), device))

fen2 = "r1bNk2r/pppp2pp/2n4n/2b1p3/2B1P3/8/PPPP1PPP/RNBQK2R w KQkq - 0 1" # white has won a queen in the opening should be white winning
print(model_evaluate_board(model, chess.Board(fen2), device))

fen3 = "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR b KQkq e3 0 1" # starting position white up a queen. should be white winning
print(model_evaluate_board(model, chess.Board(fen3), device))

fen4 = "r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 0 4" # mate in one for white and white to move
print(model_evaluate_board(model, chess.Board(fen4), device))

0.09049943834543228
0.8448002934455872
0.8988310098648071
-0.12278395891189575


In [5]:
# test2: can the model value pieces; add a pawn, add a knight, add a bishop, add a rook, add a queen and should be < <= < <
def test_piece_values(model, device):
    base_fen = "4k3/8/8/8/8/8/8/4K3 w - - 0 1"

    # Add pieces one at a time for white
    pawn_board = chess.Board(base_fen)
    pawn_board.set_piece_at(chess.E4, chess.Piece(chess.PAWN, chess.WHITE))

    knight_board = chess.Board(base_fen)
    knight_board.set_piece_at(chess.E4, chess.Piece(chess.KNIGHT, chess.WHITE))

    bishop_board = chess.Board(base_fen)
    bishop_board.set_piece_at(chess.E4, chess.Piece(chess.BISHOP, chess.WHITE))

    rook_board = chess.Board(base_fen)
    rook_board.set_piece_at(chess.E4, chess.Piece(chess.ROOK, chess.WHITE))

    queen_board = chess.Board(base_fen)
    queen_board.set_piece_at(chess.E4, chess.Piece(chess.QUEEN, chess.WHITE))

    # Evaluate
    pawn_val = model_evaluate_board(model, pawn_board, device)
    knight_val = model_evaluate_board(model, knight_board, device)
    bishop_val = model_evaluate_board(model, bishop_board, device)
    rook_val = model_evaluate_board(model, rook_board, device)
    queen_val = model_evaluate_board(model, queen_board, device)

    print("Pawn:", pawn_val)
    print("Knight:", knight_val)
    print("Bishop:", bishop_val)
    print("Rook:", rook_val)
    print("Queen:", queen_val)


test_piece_values(model, device)

Pawn: 0.0989450067281723
Knight: 0.1588871330022812
Bishop: 0.05548691004514694
Rook: 0.08581173419952393
Queen: 0.2594282627105713


In [ ]:
# test3: sanity check if symmetry is correct
def test_color_symmetry(model, device):
    fen = "r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 0 1"
    
    board = chess.Board(fen)
    flipped = board.mirror()

    val1 = model_evaluate_board(model, board, device)
    val2 = model_evaluate_board(model, flipped, device)

    print("Original:", val1)
    print("Mirrored:", val2)

test_color_symmetry(model, device)

Original: 0.12029512226581573
Mirrored: 0.0929788276553154
